In [18]:
import pandas as pd
import os
import math

import joblib
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge

from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.set_option('display.max_columns', None)

In [5]:
data = pd.read_csv('expense_data_1.csv')

In [6]:
data["RUB"] = data["INR"] * 0.86
data

,Date,Account,Category,Subcategory,Note,INR,Income/Expense,Note.1,Amount,Currency,Account.1,RUB
0,3/2/2022 10:11,CUB - online payment,Food,NaN,Brownie,50.0,Expense,NaN,50.0,INR,50.0,43.00
1,3/2/2022 10:11,CUB - online payment,Other,NaN,To lended people,300.0,Expense,NaN,300.0,INR,300.0,258.00
2,3/1/2022 19:50,CUB - online payment,Food,NaN,Dinner,78.0,Expense,NaN,78.0,INR,78.0,67.08
3,3/1/2022 18:56,CUB - online payment,Transportation,NaN,Metro,30.0,Expense,NaN,30.0,INR,30.0,25.80
4,3/1/2022 18:22,CUB - online payment,Food,NaN,Snacks,67.0,Expense,NaN,67.0,INR,67.0,57.62
...,...,...,...,...,...,...,...,...,...,...,...,...
272,11/22/2021 14:16,CUB - online payment,Food,NaN,Dinner,90.0,Expense,NaN,90.0,INR,90.0,77.40
273,11/22/2021 14:16,CUB - online payment,Food,NaN,Lunch with company,97.0,Expense,NaN,97.0,INR,97.0,83.42
274,11/21/2021 17:07,CUB - online payment,Transportation,NaN,Rapido,130.0,Expense,NaN,130.0,INR,130.0,111.80
275,11/21/2021 15:50,CUB - online payment,Food,NaN,Lunch,875.0,Expense,NaN,875.0,INR,875.0,752.50


In [ ]:
data = data.drop('Subcategory', axis=1)

In [ ]:
data = data[data['Income/Expense'] == 'Expense']
data

,Date,Account,Category,Note,INR,Income/Expense,Note.1,Amount,Currency,Account.1,RUB
0,3/2/2022 10:11,CUB - online payment,Food,Brownie,50.0,Expense,NaN,50.0,INR,50.0,43.00
1,3/2/2022 10:11,CUB - online payment,Other,To lended people,300.0,Expense,NaN,300.0,INR,300.0,258.00
2,3/1/2022 19:50,CUB - online payment,Food,Dinner,78.0,Expense,NaN,78.0,INR,78.0,67.08
3,3/1/2022 18:56,CUB - online payment,Transportation,Metro,30.0,Expense,NaN,30.0,INR,30.0,25.80
4,3/1/2022 18:22,CUB - online payment,Food,Snacks,67.0,Expense,NaN,67.0,INR,67.0,57.62
...,...,...,...,...,...,...,...,...,...,...,...
271,11/23/2021 22:53,CUB - online payment,Food,Dinner,179.0,Expense,NaN,179.0,INR,179.0,153.94
272,11/22/2021 14:16,CUB - online payment,Food,Dinner,90.0,Expense,NaN,90.0,INR,90.0,77.40
273,11/22/2021 14:16,CUB - online payment,Food,Lunch with company,97.0,Expense,NaN,97.0,INR,97.0,83.42
274,11/21/2021 17:07,CUB - online payment,Transportation,Rapido,130.0,Expense,NaN,130.0,INR,130.0,111.80


In [7]:
data["Date"] = pd.to_datetime(data["Date"])

In [11]:
if "Category" in data.columns:
    data["category"] = data["Category"]
else:
    data["category"] = data["Account"]  # fallback

# 4. PIVOT (ключевой момент)
pivot = data.pivot_table(
    index=data["Date"].dt.date,
    values="RUB",
    aggfunc="sum",
    fill_value=0
)

# 5. Общая сумма (target)
pivot["total"] = pivot.sum(axis=1)

# 6. Добавляем признаки
pivot = pivot.reset_index()

pivot["day_index"] = np.arange(len(pivot))
pivot["day_of_week"] = pd.to_datetime(pivot["Date"]).dt.weekday
pivot["prev_amount"] = pivot["total"].shift(1)

# 7. Убираем NaN
pivot = pivot.dropna()


In [13]:
pivot = pivot.drop('RUB', axis=1)
pivot["lag_1"] = pivot["total"].shift(1)
pivot["lag_3"] = pivot["total"].shift(3)
pivot["lag_7"] = pivot["total"].shift(7)
pivot["rolling_mean_3"] = pivot["total"].rolling(3).mean()
pivot["rolling_mean_7"] = pivot["total"].rolling(7).mean()
pivot = pivot.dropna()

In [14]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

In [16]:
y = pivot["total"]
X = pivot.drop(columns=["Date", "total"])

In [19]:
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# числовые признаки
num_columns = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# препроцессинг
data_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_columns),
])

pipe = Pipeline([
    ('preprocessor', data_preprocessor),
    ('model', LinearRegression()),
])

# Grid
param_grid = [

    # LinearRegression
    {
        'model': [LinearRegression()],
        'preprocessor__num': [StandardScaler(), MinMaxScaler()]
    },

    # Ridge
    {
        'model': [Ridge()],
        'preprocessor__num': [StandardScaler(), MinMaxScaler()],
        'model__alpha': [0.1, 1.0, 10.0]
    },

    # Lasso
    {
        'model': [Lasso(max_iter=10000)],
        'preprocessor__num': [StandardScaler(), MinMaxScaler()],
        'model__alpha': [0.001, 0.01, 0.1]
    },

    # RandomForest
    {
        'model': [RandomForestRegressor(random_state=42)],
        'preprocessor__num': ['passthrough'],
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 10],
        'model__min_samples_split': [2, 5]
    },

    # GradientBoosting
    {
        'model': [GradientBoostingRegressor(random_state=42)],
        'preprocessor__num': ['passthrough'],
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [3, 5]
    }
]

# GridSearch
gs = GridSearchCV(
    pipe,
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

gs.fit(X_train, y_train)

# результаты
print("Лучшие параметры:", gs.best_params_)
print("MAE:", -gs.best_score_)

# тест
y_pred = gs.predict(X_test)
print("Test MAE:", mean_absolute_error(y_test, y_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END model=LinearRegression(), preprocessor__num=StandardScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=StandardScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=StandardScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=MinMaxScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=MinMaxScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=StandardScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=StandardScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=MinMaxScaler(); total time=   0.0s
[CV] END model=LinearRegression(), preprocessor__num=MinMaxScaler(); total time=   0.0s
[CV] END model=Ridge(), model__alpha=0.1, preprocessor__num=StandardScaler(); total time=   0.0s[CV] END model=Ridge(), model__alpha=0.1

In [22]:
print(y.mean())

1160.865012658228


In [23]:
model_data = {
    "model": gs.best_estimator_,
    "features": X.columns.tolist(),
    "version": "0.1.0"
}

joblib.dump(model_data, "model.pkl")

['model.pkl']